# Use Case 2 — Realistic E-commerce Image Classification  
## Generic `langchain_openai` Vision Baseline → Local ResNet18 Fine-Tuning → Accuracy Comparison

### Business scenario

An e-commerce company has three private product catalog codes:

- Shoe → `CAT-17`
- Watch → `CAT-42`
- Bag → `CAT-81`

The generic OpenAI vision model can understand what is visible in a photograph, but it is **not told the private code mapping**.

We compare:

1. Generic OpenAI vision classification through `langchain_openai`
2. Local fine-tuning of pretrained ResNet18
3. Accuracy before and after fine-tuning

### Dataset

The folder contains realistic product photographs:

- 36 training images
- 18 evaluation images
- Shoes
- Watches
- Bags

Training and evaluation images are stored separately.

In [ ]:
pip uninstall torch torchvision torchaudio -y

In [ ]:
pip install --upgrade pip

In [ ]:
pip install torch torchvision torchaudio

## Step 1 — Install libraries

In [ ]:
%pip install -q -U langchain-openai openai python-dotenv pandas scikit-learn torch torchvision pillow matplotlib

## Step 2 — Imports and device configuration

In [ ]:
import os
import base64
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

load_dotenv()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("Training device:", DEVICE)

## Step 3 — Load the image metadata

In [ ]:
TRAIN_DIR = Path("images/train")
EVAL_DIR = Path("images/eval")

train_labels = pd.read_csv(
    "image_train_labels.csv"
)

eval_labels = pd.read_csv(
    "image_eval_labels.csv"
)

print("Training metadata:", train_labels.shape)
print("Evaluation metadata:", eval_labels.shape)

display(train_labels.head())

## Step 4 — Check class balance

In [ ]:
print("Training:")
print(train_labels["catalog_code"].value_counts())

print("\nEvaluation:")
print(eval_labels["catalog_code"].value_counts())

## Step 5 — Display realistic training and evaluation images

This verifies that the classes are visually different rather than being copies of the same synthetic image.

In [ ]:
samples = []

for product in ["shoe", "watch", "bag"]:
    train_subset = train_labels[
        train_labels["product_type"] == product
    ]

    eval_subset = eval_labels[
        eval_labels["product_type"] == product
    ]

    samples.extend([
        ("Train", train_subset.iloc[0]),
        ("Train", train_subset.iloc[6]),
        ("Eval", eval_subset.iloc[0])
    ])

fig, axes = plt.subplots(
    3,
    3,
    figsize=(11, 11)
)

for ax, (split_name, row) in zip(
    axes.ravel(),
    samples
):
    folder = (
        TRAIN_DIR
        if split_name == "Train"
        else EVAL_DIR
    )

    image = Image.open(
        folder / row["filename"]
    )

    ax.imshow(image)
    ax.set_title(
        f"{split_name}: "
        f"{row['product_type']} "
        f"({row['catalog_code']})"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

# Part A — Generic OpenAI Vision Baseline

We send each evaluation image to OpenAI using `ChatOpenAI`.

The model sees only three opaque company codes:
- `CAT-17`
- `CAT-42`
- `CAT-81`

It is not told which product belongs to which code.

## Step 6 — Convert local images to Base64

In [ ]:
def image_to_data_url(path):
    encoded = base64.b64encode(
        Path(path).read_bytes()
    ).decode("utf-8")

    return f"data:image/jpeg;base64,{encoded}"

## Step 7 — Configure the generic OpenAI vision model

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

assert os.getenv("OPENAI_API_KEY"), (
    "Create .env with OPENAI_API_KEY=your_key"
)

OPENAI_VISION_MODEL = os.getenv(
    "OPENAI_VISION_MODEL",
    "gpt-4.1-mini"
)

vision_llm = ChatOpenAI(
    model=OPENAI_VISION_MODEL,
    temperature=0
)

VISION_SYSTEM_PROMPT = (
    "You are an e-commerce image classifier. "
    "Return exactly one private catalog code: "
    "CAT-17, CAT-42, or CAT-81. "
    "Return the code only."
)

## Step 8 — Create a generic vision prediction function

In [ ]:
VALID_IMAGE_CODES = [
    "CAT-17",
    "CAT-42",
    "CAT-81"
]

def extract_image_code(text):
    text = str(text).upper()

    for code_value in VALID_IMAGE_CODES:
        if code_value in text:
            return code_value

    return "INVALID"

def generic_vision_predict(image_path):
    message = HumanMessage(content=[
        {
            "type": "text",
            "text": (
                "Classify this product image. "
                "Return only CAT-17, CAT-42, or CAT-81."
            )
        },
        {
            "type": "image_url",
            "image_url": {
                "url": image_to_data_url(
                    image_path
                ),
                "detail": "low"
            }
        }
    ])

    response = vision_llm.invoke([
        SystemMessage(
            content=VISION_SYSTEM_PROMPT
        ),
        message
    ])

    return extract_image_code(
        response.content
    )

## Step 9 — Run the generic vision model on evaluation images

In [ ]:
generic_predictions = []

for i, row in eval_labels.iterrows():

    prediction = generic_vision_predict(
        EVAL_DIR / row["filename"]
    )

    generic_predictions.append(
        prediction
    )

    print(
        f"{i+1:02d}. "
        f"Actual={row['catalog_code']} | "
        f"Generic={prediction}"
    )

## Step 10 — Calculate generic vision accuracy

In [ ]:
generic_vision_accuracy = accuracy_score(
    eval_labels["catalog_code"],
    generic_predictions
)

generic_results = eval_labels.copy()
generic_results[
    "generic_openai_prediction"
] = generic_predictions

display(generic_results)

print(
    f"Generic OpenAI vision accuracy: "
    f"{generic_vision_accuracy:.2%}"
)

# Part B — Local Image Fine-Tuning with ResNet18

ResNet18 is a pretrained convolutional neural network.

Instead of training a CNN from scratch, we use **transfer learning**:

1. Load pretrained ResNet18.
2. Freeze most pretrained layers.
3. Replace the final classification layer.
4. Train the new layer for our three private classes.
5. Optionally unfreeze the last residual block for light fine-tuning.

This is a real image fine-tuning workflow.

## Step 11 — Define class IDs

In [ ]:
CLASS_TO_ID = {
    "CAT-17": 0,
    "CAT-42": 1,
    "CAT-81": 2
}

ID_TO_CLASS = {
    value: key
    for key, value in CLASS_TO_ID.items()
}

print(CLASS_TO_ID)

## Step 12 — Define image transformations

Training transformations add natural variation:
- random crop
- horizontal flip
- small rotation
- color variation

Evaluation uses deterministic resize/crop only.

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(
        224,
        scale=(0.75, 1.0)
    ),
    transforms.RandomHorizontalFlip(
        p=0.4
    ),
    transforms.RandomRotation(
        degrees=8
    ),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Step 13 — Create a custom image Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ProductImageDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform
    ):
        self.dataframe = dataframe.reset_index(
            drop=True
        )
        self.image_dir = Path(
            image_dir
        )
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = (
            self.image_dir
            / row["filename"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        image = self.transform(
            image
        )

        label = CLASS_TO_ID[
            row["catalog_code"]
        ]

        return image, label

train_dataset = ProductImageDataset(
    train_labels,
    TRAIN_DIR,
    train_transform
)

eval_dataset = ProductImageDataset(
    eval_labels,
    EVAL_DIR,
    eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=8,
    shuffle=False
)

print("Training images:", len(train_dataset))
print("Evaluation images:", len(eval_dataset))

## Step 14 — Load pretrained ResNet18

In [ ]:
from torchvision.models import (
    resnet18,
    ResNet18_Weights
)
import torch.nn as nn

weights = ResNet18_Weights.DEFAULT

resnet_model = resnet18(
    weights=weights
)

print(resnet_model.fc)

## Step 15 — Freeze the pretrained backbone

Initially we keep the learned image features fixed and train only the final classifier.

In [ ]:
for parameter in resnet_model.parameters():
    parameter.requires_grad = False

number_of_features = (
    resnet_model.fc.in_features
)

resnet_model.fc = nn.Linear(
    number_of_features,
    3
)

resnet_model = resnet_model.to(
    DEVICE
)

print(resnet_model.fc)

## Step 16 — Configure loss and optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    resnet_model.fc.parameters(),
    lr=1e-3
)

EPOCHS = 8

print("Epochs:", EPOCHS)

## Step 17 — Fine-tune the classifier

In [ ]:
training_losses = []
training_accuracies = []

for epoch in range(EPOCHS):

    resnet_model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = resnet_model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = (
        running_loss
        / len(train_loader)
    )

    epoch_accuracy = (
        correct / total
    )

    training_losses.append(
        epoch_loss
    )

    training_accuracies.append(
        epoch_accuracy
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss={epoch_loss:.4f} | "
        f"Train Accuracy="
        f"{epoch_accuracy:.2%}"
    )

## Step 18 — Plot training loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    range(1, EPOCHS + 1),
    training_losses,
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet18 Fine-Tuning Loss")
plt.grid(True)
plt.show()

## Step 19 — Plot training accuracy

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    range(1, EPOCHS + 1),
    training_accuracies,
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Training Accuracy")
plt.title("ResNet18 Training Accuracy")
plt.grid(True)
plt.show()

## Step 20 — Evaluate ResNet18 on unseen images

In [ ]:
resnet_model.eval()

local_predictions = []
actual_labels = []

with torch.no_grad():

    for images, labels in eval_loader:

        images = images.to(DEVICE)

        outputs = resnet_model(
            images
        )

        predicted_ids = torch.argmax(
            outputs,
            dim=1
        ).cpu().tolist()

        local_predictions.extend(
            ID_TO_CLASS[i]
            for i in predicted_ids
        )

        actual_labels.extend(
            ID_TO_CLASS[i]
            for i in labels.tolist()
        )

local_image_accuracy = accuracy_score(
    actual_labels,
    local_predictions
)

print(
    f"Fine-tuned ResNet18 accuracy: "
    f"{local_image_accuracy:.2%}"
)

## Step 21 — Compare generic OpenAI vs fine-tuned ResNet18

In [ ]:
comparison = eval_labels.copy()

comparison[
    "generic_openai_prediction"
] = generic_predictions

comparison[
    "fine_tuned_resnet18"
] = local_predictions

display(comparison)

print(
    f"Generic OpenAI baseline : "
    f"{generic_vision_accuracy:.2%}"
)

print(
    f"Fine-tuned ResNet18     : "
    f"{local_image_accuracy:.2%}"
)

print(
    f"Accuracy improvement    : "
    f"{local_image_accuracy-generic_vision_accuracy:.2%}"
)

## Step 22 — Classification report

In [ ]:
print(classification_report(
    actual_labels,
    local_predictions,
    labels=[
        "CAT-17",
        "CAT-42",
        "CAT-81"
    ],
    zero_division=0
))

## Step 23 — Confusion matrix

In [ ]:
cm = confusion_matrix(
    actual_labels,
    local_predictions,
    labels=[
        "CAT-17",
        "CAT-42",
        "CAT-81"
    ]
)

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "CAT-17",
        "CAT-42",
        "CAT-81"
    ]
)

display_cm.plot()
plt.title(
    "Fine-Tuned ResNet18 "
    "Confusion Matrix"
)
plt.show()

## Step 24 — Display predictions with images

In [ ]:
fig, axes = plt.subplots(
    3,
    6,
    figsize=(16, 9)
)

for ax, (_, row), prediction in zip(
    axes.ravel(),
    eval_labels.iterrows(),
    local_predictions
):
    image = Image.open(
        EVAL_DIR / row["filename"]
    )

    ax.imshow(image)

    ax.set_title(
        f"Actual: {row['catalog_code']}\n"
        f"Pred: {prediction}"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

## Step 25 — Save the locally fine-tuned ResNet18

In [ ]:
MODEL_FILE = (
    "fine_tuned_ecommerce_resnet18.pth"
)

torch.save(
    {
        "model_state_dict":
            resnet_model.state_dict(),
        "class_to_id":
            CLASS_TO_ID
    },
    MODEL_FILE
)

print("Saved:", MODEL_FILE)